# Otvoreni tok: predvidi → izračunaj → provjeri

Za pravokutni kanal specifična energija po jedinici širine glasi

\[
E(y)=y+\frac{q^2}{2gy^2}.
\]

Ista energija iznad minimuma ima dvije alternativne dubine, ali rubni uvjeti odlučuju koja se grana ostvaruje. Hidraulički skok povezuje nadkritični i podkritični tok uz disipaciju energije.

## Predvidi

1. Skiciraj \(E(y)\), označi kritičnu dubinu i dvije grane.
2. Koja alternativna dubina ima \(Fr>1\)?
3. Kako nesigurnost mjerenja uzvodne dubine i brzine prelazi na procijenjenu spregnutu dubinu skoka?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
g, q = 9.81, 1.5

def energy(y, discharge_per_width=q):
    return y + discharge_per_width**2/(2*g*y**2)

def froude(y, discharge_per_width=q):
    return discharge_per_width/(y*np.sqrt(g*y))

def bisect_target(function, a, b, target, tolerance=1e-12, max_iter=100):
    fa, fb = function(a)-target, function(b)-target
    if fa*fb > 0:
        raise ValueError("Interval ne omeđuje korijen.")
    history = []
    for iteration in range(max_iter):
        m = 0.5*(a+b)
        fm = function(m)-target
        history.append((iteration, m, fm))
        if abs(fm) < tolerance:
            return m, np.asarray(history)
        if fa*fm <= 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    raise RuntimeError("Bisekcija nije konvergirala.")

yc = (q**2/g)**(1/3)
Emin = 1.5*yc
E_target = 1.20
y_shallow, hist_shallow = bisect_target(energy, 0.05, yc, E_target)
y_deep, hist_deep = bisect_target(energy, yc, 4.0, E_target)
print(f"yc={yc:.4f} m; Emin={Emin:.4f} m")
print(f"Alternativne dubine: {y_shallow:.4f} m (Fr={froude(y_shallow):.3f}) i {y_deep:.4f} m (Fr={froude(y_deep):.3f})")


## Izračunaj: hidraulički skok i mjerna nesigurnost

Za pravokutni kanal spregnuta dubina iz uzvodne dubine \(y_1\) i brzine \(v_1\) jest

\[
y_2=\frac{y_1}{2}\left(\sqrt{1+8Fr_1^2}-1\right).
\]

Monte Carlo uzorkovanje uspoređujemo s lokalnom linearizacijom dobivenom centriranim razlikama. Time razlikujemo nesigurnost ulaza od fizikalnog gubitka energije kroz skok.


In [ ]:
def conjugate_depth(y1, v1):
    Fr1 = v1/np.sqrt(g*y1)
    return 0.5*y1*(np.sqrt(1+8*Fr1**2)-1)

y1, v1 = 0.250, 6.00
y2 = conjugate_depth(y1, v1)
q_jump = y1*v1
Fr1 = v1/np.sqrt(g*y1)
energy_loss = (y2-y1)**3/(4*y1*y2)
momentum_1 = y1**2/2 + q_jump**2/(g*y1)
momentum_2 = y2**2/2 + q_jump**2/(g*y2)

sigma_y, sigma_v = 0.003, 0.08
dy = sigma_y*1e-3
dv = sigma_v*1e-3
grad_y = (conjugate_depth(y1+dy, v1)-conjugate_depth(y1-dy, v1))/(2*dy)
grad_v = (conjugate_depth(y1, v1+dv)-conjugate_depth(y1, v1-dv))/(2*dv)
u_y2_linear = np.sqrt((grad_y*sigma_y)**2 + (grad_v*sigma_v)**2)

rng = np.random.default_rng(20260804)
n_samples = 40_000
y1_mc = rng.normal(y1, sigma_y, n_samples)
v1_mc = rng.normal(v1, sigma_v, n_samples)
y2_mc = conjugate_depth(y1_mc, v1_mc)
u_y2_mc = np.std(y2_mc, ddof=1)
interval = np.quantile(y2_mc, [0.025, 0.975])

print(f"Skok: Fr1={Fr1:.3f}, y2={y2:.4f} m, ΔE={energy_loss:.4f} m")
print(f"u(y2) linearno={u_y2_linear:.5f} m; Monte Carlo={u_y2_mc:.5f} m")
print(f"95 %-tni interval y2=[{interval[0]:.4f}, {interval[1]:.4f}] m")


## Provjeri

Alternativne dubine moraju vratiti istu specifičnu energiju i ležati s različitih strana kritičnog stanja. Za skok provjeravamo očuvanje funkcije količine gibanja te slaganje dvaju postupaka propagacije nesigurnosti.


In [ ]:
assert np.isclose(energy(y_shallow), E_target, rtol=1e-11)
assert np.isclose(energy(y_deep), E_target, rtol=1e-11)
assert froude(y_shallow) > 1 and froude(y_deep) < 1
assert np.isclose(momentum_1, momentum_2, rtol=1e-12)
assert y2 > y1 and energy_loss > 0
assert abs(u_y2_mc/u_y2_linear-1) < 0.05

ys = np.linspace(0.12, 2.0, 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ys, energy(ys), color="#256d85", lw=2)
axes[0].axhline(E_target, color="#7a8a96", ls="--")
axes[0].plot([y_shallow, y_deep], [E_target, E_target], "o", color="#b43c35")
axes[0].plot(yc, Emin, "s", color="#2d7d46", label="kritično stanje")
axes[0].set(xlabel="dubina y (m)", ylabel="specifična energija E (m)", title="Dvije grane iste energije")
axes[0].legend()
axes[1].hist(y2_mc, bins=55, color="#7cb5d6", edgecolor="white")
axes[1].axvline(y2, color="#b43c35", lw=2, label="nominalno")
axes[1].set(xlabel="spregnuta dubina y2 (m)", ylabel="broj uzoraka", title="Nesigurnost procjene skoka")
axes[1].legend()
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Jednakost specifične energije ne bira ostvarenu dubinu; tu odluku donose rubni uvjeti i smjer informacija u nadkritičnom ili podkritičnom toku. Kroz skok se približno čuva bilanca količine gibanja, dok se mehanička energija disipira.
